# 第1章 LangChain 基礎

LangChain経由で Amazon Bedrock のLLMを呼び出し、`Prompt → Model → Parser` のチェーン(LCEL)を体験します。

**所要時間**: 約20分

## 0. 準備 - 環境変数の読み込み

`.env` から `AWS_PROFILE` / `AWS_REGION` / モデルIDを取り込みます。

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

AWS_REGION = os.environ["AWS_REGION"]
CHAT_MODEL_ID = os.environ["BEDROCK_CHAT_MODEL_ID"]

print("region:", AWS_REGION)
print("chat model:", CHAT_MODEL_ID)
print("profile:", os.environ.get("AWS_PROFILE"))

## 1. Hello Bedrock

`ChatBedrockConverse` は Bedrock の **Converse API** を使うチャットモデルクラスです。
ツール呼び出しなどがモデル横断で扱えるため、本教材では基本これを使います。

In [ ]:
from langchain_aws import ChatBedrockConverse

llm = ChatBedrockConverse(
    model=CHAT_MODEL_ID,
    region_name=AWS_REGION,
    temperature=0,
)

response = llm.invoke("こんにちは。あなたは何ができますか? 30字程度で答えてください。")
print(response.content)

応答テキストが返ってくれば、Bedrockへの接続OKです。

`response` は `AIMessage` オブジェクト。`.content` がテキスト本体です。

## 2. PromptTemplate でプロンプトを構造化

プロンプトをハードコードせず、テンプレ化して変数で差し込めるようにします。
- `system`: モデルの役割や守ってほしいルール
- `human`: ユーザーからの入力

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "あなたは{role}です。簡潔に日本語で答えてください。"),
    ("human", "{question}"),
])

messages = prompt.format_messages(role="プログラミング講師", question="LangChainを一言で説明して")
for m in messages:
    print(f"[{m.type}] {m.content}")

In [ ]:
response = llm.invoke(messages)
print(response.content)

## 3. LCEL: パイプでチェーンを組む

LangChain Expression Language (LCEL) は **コンポーネントを `|` で繋いでパイプライン化** する書き方です。

```
prompt  |  llm  |  StrOutputParser
   ↑        ↑          ↑
 入力整形  推論     文字列に整形
```

`StrOutputParser` を最後に挟むと、`AIMessage` ではなく **文字列がそのまま返る** ので扱いやすくなります。

In [ ]:
from langchain_core.output_parsers import StrOutputParser

chain = prompt | llm | StrOutputParser()

answer = chain.invoke({
    "role": "料理研究家",
    "question": "カレーを美味しく作る3つのコツを教えて",
})

print(answer)

## 4. ストリーミングで応答を受け取る

LCELで組んだチェーンは、そのまま `.stream()` でトークン単位の応答が得られます。
UIに逐次表示したい時などに便利です。

In [ ]:
for chunk in chain.stream({
    "role": "歴史の先生",
    "question": "日本の応仁の乱について100字くらいで説明して",
}):
    print(chunk, end="", flush=True)

## まとめ

- `ChatBedrockConverse` で Bedrock のチャットモデルを呼べる
- `ChatPromptTemplate` で入力を構造化できる
- `prompt | llm | parser` のLCELで、推論パイプラインを宣言的に組める
- 同じチェーンが `.invoke()` / `.stream()` 両対応

次は [02 RAG](../02_rag/rag.ipynb) で、外部文書を参照する仕組みを作ります。